# 🚀 DataMate — Zalo AI Challenge 2026: Public Test Inference

Notebook hoàn chỉnh giúp bạn chạy toàn bộ pipeline suy luận trên tập **`public_test`** (1,000 câu hỏi) bằng **GPU Google Colab** (A100 / L4 / T4) và xuất file **`submission.jsonl`** chuẩn để nộp lên bảng xếp hạng.

> 💡 **Lưu ý:** Trước khi chạy, hãy chọn GPU: **Runtime** → **Change runtime type** → chọn **A100 GPU** (hoặc L4 / T4).

In [35]:
# 1.1 Kiểm tra thông tin GPU đang kết nối
!nvidia-smi

Thu Oct 29 14:03:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [36]:
# 1.2 Cài đặt các thư viện phụ thuộc cần thiết
!pip install -q duckdb>=1.1.0 pandas>=2.2.0 openpyxl python-calamine
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 bitsandbytes>=0.43.0
!pip install -q tqdm loguru unidecode regex

---  
### 2. Kết nối Google Drive & Đồng bộ Code GitHub

In [37]:
# 2.1 Mount Google Drive và trỏ vào thư mục làm việc
from google.colab import drive
drive.mount('/content/drive')

# Chuyển vào thư mục chứa dữ liệu trên Google Drive
%cd /content/drive/MyDrive/ZALO_AI_Challenge

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ZALO_AI_Challenge


In [38]:
# 🔄 2.2 ĐỒNG BỘ CODE MỚI NHẤT TỪ GITHUB (NHÁNH DEV)
# Bấm chạy ô này bất kỳ lúc nào để tự động cập nhật run_public_test.py, src/, prebuild_cache.py, ...
import os

REPO_URL = "https://github.com/JimmyK300/zalo-vecna.git"
BRANCH = "dev"

print(f"[*] Đang đồng bộ mã nguồn từ GitHub (nhánh {BRANCH})...")
if not os.path.exists(".git"):
    !git init
    !git config core.filemode false
    !git remote add origin {REPO_URL}
else:
    !git remote set-url origin {REPO_URL}

!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git reset --hard origin/{BRANCH}

print("\n✅ Code đã cập nhật thành công! Commit hiện tại:")
!git log -1 --oneline

[*] Đang đồng bộ mã nguồn từ GitHub (nhánh dev)...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 8.95 KiB | 127.00 KiB/s, done.
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
   994dbc6..000bed8  dev        -> origin/dev
branch 'dev' set up to track 'origin/dev'.
Reset branch 'dev'
Your branch is up to date with 'origin/dev'.
HEAD is now at 000bed8 Enhance multi-pass self-healing execution, SQLite table aliases, mechanic name parsing, and prompt instructions

✅ Code đã cập nhật thành công! Commit hiện tại:
000bed8 (HEAD -> dev, origin/dev) Enhance multi-pass self-healing execution, SQLite table aliases, mechanic name parsing, and prompt instructions


---  
### 3. Tăng Tốc DuckDB — Tiền Xử Lý Cache (Chạy 1 lần duy nhất)
Chuyển đổi dữ liệu 13 doanh nghiệp (`dev_001` → `dev_013`) sang database `.duckdb` nén.  
👉 Giúp DuckDB **nạp dữ liệu tức thì trong 0.01 giây** thay vì phải đọc và phân tích file Excel qua Google Drive mỗi lần suy luận!

In [39]:
# ⚡ 3.1 Tiền xử lý DuckDB Cache cho toàn bộ 13 doanh nghiệp (mất ~1 phút, chỉ cần chạy 1 lần)
!python prebuild_cache.py --data_dir public_test --overwrite

[*] Found 15 businesses in public_test/data.
[*] Pre-building high-speed DuckDB disk cache (.duckdb)...
Caching businesses: 100% 15/15 [02:00<00:00,  8.04s/it]

[+] All 15 businesses cached in 120.57 seconds!
[+] Subsequent pipeline runs will load in ~0.01s per business!


---  
### 4. Chạy Thử Nghiệm Sanity Check (16 câu đầu)
Kiểm tra mô hình Qwen, kết nối DuckDB, Auto-UNION views và bộ sinh mã Python hoạt động chuẩn xác trước khi chạy toàn bộ 1,000 câu.

In [40]:
# 4.1 Chạy thử 16 câu đầu (tự động xóa sample cũ để kiểm tra code mới nhất)
!rm -f submission_sample.jsonl
!rm -rf result/logs

# Chạy tốc độ cao với FP16 và Batch size 16 trên A100 GPU:
!python run_public_test.py \
    --data_dir public_test \
    --output submission_sample.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --fp16 \
    --batch_size 16 \
    --limit 16 \
    --overwrite

# In kết quả sinh ra của 16 câu
!cat submission_sample.jsonl

=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: False)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 75.97it/s]
[+] Model loaded successfully!
[*] Processing first 16 questions only.
Inferring: 100% 16/16 [01:00<00:00,  3.77s/it]

[+] Inference completed successfully!
[+] Output saved to: submission_sample.jsonl
[+] Logs saved to: result/logs
{"id": "q_008fdebb6ec9", "type": "string", "answer": "Trà đào cam sả (M)"}
{"id": "q_009187cd2960", "type": "number", "answer": 80}
{"id": "q_00bf8df3ea0b", "type": "number", "answer": 4750000}
{"id": "q_00c4e24fa228", "type": "number", "answer": 16708000}
{"id": "q_013b4ff35811", "type": "number", "answer": 208691600}
{"id": "q_014cdcc1da6e", "type": "number", "answer": 19487000}
{"id": "q_01ac0fc9f2a2", "type": "number", "answer": 87956.57}
{"id": 

In [41]:
# 🩺 4.2 Chẩn đoán & Xem chi tiết log suy luận (Code SQL/Python sinh ra)
import glob, json, os

log_files = sorted(glob.glob("result/logs/*.json"))
print(f"[+] Đã tìm thấy {len(log_files)} file log trong result/logs/")

# 1. In nhanh nếu có câu nào bị abstain
abstain_count = 0
for f in log_files:
    d = json.load(open(f, encoding="utf-8"))
    if d.get("final_submission", {}).get("type") == "abstain":
        abstain_count += 1
        print(f"\n⚠️ [ABSTAIN] Câu: {d['id']} | {d.get('question')}")
        print("CODE:", (d.get("generated_code") or "")[:600])
        print("LỖI:", d.get("exec_error") or d.get("error"))

if abstain_count == 0 and log_files:
    print("🎉 TUYỆT VỜI! 100% câu hỏi đều trả về kết quả hợp lệ, không có câu nào bị abstain!")

# 2. Xem chi tiết mẫu câu đầu tiên để kiểm tra code sinh ra
if log_files:
    sample_log = json.load(open(log_files[0], encoding="utf-8"))
    print(f"\n--- Mẫu Log chi tiết ({os.path.basename(log_files[0])}) ---")
    print(json.dumps(sample_log, indent=2, ensure_ascii=False))

[+] Đã tìm thấy 16 file log trong result/logs/

⚠️ [ABSTAIN] Câu: q_027f5441831d | Tháng 8/2025 thợ Đoàn Bảo Phúc lãnh bao nhiêu tiền huê hồng?
CODE: query = """
WITH thong_ke AS (
    SELECT 
        s.tho,
        SUM(s.tien) * ANY_VALUE(d.hoa_hong) / 100.0 AS hoa_hong
    FROM 
        so_sua_xe s
    JOIN 
        danh_sach_tho d ON d.ho_ten ILIKE '%Doan Bảo Phuc%'
    WHERE 
        s.ngay BETWEEN '2025-08-01' AND '2025-08-31'
    GROUP BY 
        s.tho
)
SELECT 
    hoa_hong
FROM 
    thong_ke;
"""

result = con.execute(query).fetchone()[0]
LỖI: 'NoneType' object is not subscriptable

⚠️ [ABSTAIN] Câu: q_033e37dde4d5 | Tuần này bán được tổng cộng bao nhiêu gói rồi ha?
CODE: result = {
    "type": "number",
    "answer": con.execute("""
        SELECT SUM(CASE WHEN dvt ILIKE '%gói%' THEN sl_quy_doi ELSE sl END) AS total_packages
        FROM chi_tiet_don
        WHERE ngay_lap BETWEEN '2026-06-22' AND '2026-06-25'
          AND COALESCE(trang_thai, '') NOT ILIKE '%huỷ%'
    """).

---  
### 5. Chạy Toàn Bộ 1,000 Câu Hỏi (Full Public Test)
Lệnh này sẽ duyệt toàn bộ 1,000 câu hỏi và lưu liên tục vào file `submission.jsonl`.  
Hệ thống đã tích hợp tính năng **Tự Động Tiếp Tục (Auto-Resume)**: nếu mạng bị ngắt hoặc phiên Colab dừng giữa chừng, bạn chỉ cần bấm chạy lại ô này, script sẽ tự nhận diện các câu đã hoàn thành và chạy tiếp mà không cần làm lại từ đầu.

In [ ]:
# 🚀 5.1 CHẠY TỐC ĐỘ CAO TRÊN A100 GPU (Khuyến nghị cho Colab A100: ~15-20 phút cho 1,000 câu)
!python run_public_test.py \
    --data_dir public_test \
    --output submission.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --fp16 \
    --batch_size 16

# 💡 Lưu ý: Nếu dùng GPU T4 Colab Free, đổi sang cấu hình 4-bit bên dưới:
# !python run_public_test.py --data_dir public_test --output submission.jsonl --model_id Qwen/Qwen2.5-Coder-7B-Instruct --load_in_4bit --batch_size 4

---  
### 6. Kiểm Tra Thống Kê & Tải Kết Quả Về Máy
Kiểm tra chất lượng và đủ 1,000 dòng trước khi nộp lên cổng Zalo AI Challenge.

In [ ]:
# 6.1 Kiểm tra số lượng dòng và phân bố kiểu dữ liệu đầu ra
import os, json
from collections import Counter

if os.path.exists("submission.jsonl"):
    lines = []
    with open("submission.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                lines.append(json.loads(line))

    print(f"📊 Tổng số câu đã hoàn thành: {len(lines)} / 1000")
    types = Counter([x.get("type") for x in lines])
    print("\nPhân bố các loại đáp án:")
    for t, c in types.most_common():
        pct = (c / len(lines)) * 100
        print(f"  • {t:10}: {c:4} câu ({pct:5.1f}%)")
else:
    print("Chưa tìm thấy file submission.jsonl. Vui lòng chạy ô 5.1 trước!")

In [ ]:
# 📥 6.2 Tải file submission.jsonl về máy tính để nộp lên cổng thi Zalo
from google.colab import files

if os.path.exists("submission.jsonl"):
    files.download("submission.jsonl")
    print("✅ Đang tải file submission.jsonl về máy...")
else:
    print("Chưa có file submission.jsonl để tải về.")